# Prediction 1 — Seed-to-seed differences in learned group structure are gauge

**Claim under test.** Two independently trained networks that learn the same group
structure differ (in their structured weights) only by a gauge transformation, up to
the floor set by gauge-invariant mismatch. Corollary tested in Part C: an intervention
fit on one seed transfers to another *only after* gauge alignment.

**Setup.** Modular addition (a+b) mod p, Gromov-style 2-layer quadratic MLP, trained
past grokking at multiple seeds. Analysis object: the a-side embedding X ∈ R^{p×N}.

---
## PRE-REGISTERED PLAN — frozen before running.

**Gauge group.** (Model-space orthogonal Q ∈ O(N)) × (token-side per-mode commutant).
With N ≫ p the Procrustes Q absorbs token-side phases, so alignment is implemented as
orthogonal Procrustes; per-mode phases are reported separately for interpretability.

**Stable (gauge-invariant) quantities — predicted to match across seeds:**
per-mode energies, per-mode singular values (incl. circularity s2/s1), active-mode set.
**Unstable (gauge-dependent) — predicted NOT to match:** model-space mode planes
(principal-angle overlap ≈ 0 raw), per-mode phases.

**Primary test (P1a).** aligned_residual² ≈ spectral_floor ≪ raw_difference, where
spectral_floor = Σ_k Σ_i (s_i^A(k) − s_i^B(k))² is what no gauge map can fix.
**Null (P1b).** A singular-value-matched random matrix must NOT align comparably
(otherwise the test is vacuous). Require real-pair residual < null residual / 5.
**Transfer (P1c).** Ablating the model-space planes of the top-5 modes, measured as
mean correct-class logit-margin damage: aligned-transfer damage / self damage ≥ 0.8;
raw-transfer / self ≤ 0.3. Margin, not accuracy: the dense solution is redundant
enough that small ablations don't flip argmax (verified — accuracy is insensitive
even for the positive control, so accuracy would be an invalid readout here).

**Known limitation (stated up front).** The quadratic MLP converges to the DENSE
symmetric Fourier solution (all (p−1)/2 modes, near-equal energy, circular). Sparse
frequency selection (Chughtai-style seed variability of WHICH irreps) does not occur
in this architecture, so this notebook tests gauge-equivalence in the
maximally-shared-structure regime. The sparse regime (1-layer transformer) is a
follow-up; the multiplicity-scaling claim is NOT testable at all in Z/p (every mode
has multiplicity 1) and is out of scope here.


In [ ]:
SMOKE = True   # True: p=59, 2 seeds. False: p=97, 5 seeds, all pairs.
import numpy as np
P            = 59 if SMOKE else 97
SEEDS        = [0, 1] if SMOKE else [0, 1, 2, 3, 4]
WIDTH        = 512
TRAIN_FRAC   = 0.6
LR, WD       = 2e-3, 1.0
MAX_STEPS    = 30000
DEVICE       = "cpu"   # set "cuda" if available; CPU is fine at these sizes
print(f"p={P} seeds={SEEDS} width={WIDTH}")

In [ ]:
"""Core for Predictions 1 & 2: seed-gauge analysis of learned modular-arithmetic
embeddings, and commutant decomposition of fine-tuning changes.

Definitions (frozen):

- Embedding X in R^{p x N}: row t = model-space vector of token t.
- Real Fourier modes B_k over Z/p (from fourier_modes): C_k = B_k^T X, the
  mode-k content (2 x N, or 1 x N for the Nyquist mode).
- Complex form of a 2-row mode content: z = C[0] + i C[1] in C^N. The token-side
  commutant of the Z/p action on mode k is C^* acting as z -> w z
  (rotation+scale). The commutant ORBIT TANGENT at z is span_R{z, iz}.

GAUGE-INVARIANT per mode: energy ||C_k||_F^2 and singular values of C_k.
GAUGE-DEPENDENT: everything else (phase, model-space plane).

Alignment test (Prediction 1): the full gauge group between two seeds is
(model-space orthogonal Q) x (token-side per-mode commutant). With large N the
model-space Q absorbs token-side phases, so alignment = orthogonal Procrustes
X_A Q ~ X_B. No gauge map can change per-mode singular values, so
    residual^2 >= spectral_floor := sum_k sum_i (s_i^A(k) - s_i^B(k))^2.
Prediction: residual^2 ~ spectral_floor << raw difference. Falsifier: a large
gap between residual and floor = genuine non-gauge structural difference.

Change decomposition (Prediction 2): for small change D_k = C_k' - C_k, the
commutant-tangent fraction is ||proj_{span_R{z,iz}} d||^2 / ||d||^2 where
d = D_k in complex form. Null level for an unstructured change: 1/N.
"""
import numpy as np


def fourier_modes(n):
    j = np.arange(n)
    modes = [(0, np.ones((n, 1)) / np.sqrt(n))]
    for k in range(1, n // 2 + 1):
        c = np.cos(2 * np.pi * k * j / n)
        s = np.sin(2 * np.pi * k * j / n)
        M = np.column_stack([c, s]) if (2 * k != n) else c[:, None]
        Q, _ = np.linalg.qr(M)
        modes.append((k, Q))
    assert sum(B.shape[1] for _, B in modes) == n
    return modes


# ---------------- gauge-invariant / gauge-dependent inventory ----------------
def mode_table(X, modes):
    """Per-mode: energy, singular values, complex content z (or None for 1-dim
    modes)."""
    rows = []
    for k, B in modes:
        C = B.T @ X
        s = np.linalg.svd(C, compute_uv=False)
        z = (C[0] + 1j * C[1]) if C.shape[0] == 2 else None
        rows.append(dict(k=k, dim=C.shape[0], energy=float(np.sum(C ** 2)),
                         svals=s, z=z, C=C))
    return rows

def active_modes(table, factor=3.0):
    """Modes whose energy exceeds `factor` x the uniform share (excl. mode 0)."""
    tot = sum(r['energy'] for r in table if r['k'] > 0)
    n_modes = sum(1 for r in table if r['k'] > 0)
    thr = factor * tot / n_modes
    return sorted(r['k'] for r in table if r['k'] > 0 and r['energy'] > thr)


# ---------------- Prediction 1: alignment with spectral floor -----------------
def procrustes_align(XA, XB):
    """Q = argmin_{Q in O(N)} ||XA Q - XB||_F. Returns Q and residual^2."""
    U, s, Vt = np.linalg.svd(XA.T @ XB)
    Q = U @ Vt
    res2 = float(np.linalg.norm(XA @ Q - XB) ** 2)
    return Q, res2

def spectral_floor(tabA, tabB):
    """sum over modes of squared singular-value mismatch (gauge cannot fix)."""
    tot = 0.0
    for a, b in zip(tabA, tabB):
        sa, sb = a['svals'], b['svals']
        m = max(len(sa), len(sb))
        sa = np.pad(sa, (0, m - len(sa))); sb = np.pad(sb, (0, m - len(sb)))
        tot += float(np.sum((sa - sb) ** 2))
    return tot

def seed_pair_report(XA, XB, modes):
    tabA, tabB = mode_table(XA, modes), mode_table(XB, modes)
    raw = float(np.linalg.norm(XA - XB) ** 2)
    Q, res2 = procrustes_align(XA, XB)
    floor = spectral_floor(tabA, tabB)
    scale = float(np.linalg.norm(XA) ** 2 + np.linalg.norm(XB) ** 2) / 2
    return dict(raw=raw, aligned_residual=res2, spectral_floor=floor,
                raw_frac=raw / scale, aligned_frac=res2 / scale,
                floor_frac=floor / scale, Q=Q,
                active_A=active_modes(tabA), active_B=active_modes(tabB))


# ---------------- Prediction 2: commutant-tangent change decomposition -------
def commutant_change_decomposition(X_base, X_new, modes):
    """Per mode: split the CHANGE into commutant-tangent part (rotation+scale
    of the base content) vs orthogonal residual. Null level = 1/N."""
    out = []
    for k, B in modes:
        Cb, Cn = B.T @ X_base, B.T @ X_new
        D = Cn - Cb
        dE = float(np.sum(D ** 2))
        if Cb.shape[0] == 2:
            z = Cb[0] + 1j * Cb[1]
            d = D[0] + 1j * D[1]
            nz = np.linalg.norm(z)
            if nz < 1e-12 or dE < 1e-18:
                frac = np.nan
            else:
                u = z / nz
                # proj of d onto span_R{u, iu} = <u,d> u (complex inner product)
                coef = np.vdot(u, d)
                frac = float(np.abs(coef) ** 2 / np.sum(np.abs(d) ** 2))
        elif k == 0:
            frac = np.nan
        else:  # Nyquist: commutant is R^*, tangent = span_R{c}
            c = Cb[0]; d = D[0]
            nc = np.linalg.norm(c)
            frac = float((c @ d) ** 2 / (nc ** 2 * (d @ d))) if nc > 1e-12 and dE > 1e-18 else np.nan
        out.append(dict(k=k, change_energy=dE, commutant_frac=frac,
                        energy_base=float(np.sum(Cb ** 2)),
                        energy_new=float(np.sum(Cn ** 2))))
    return out


# ---------------- grokking trainer (modular addition, quadratic MLP) ---------
def train_grokking(p, seed, width=512, train_frac=0.6, lr=1e-3, wd=0.5,
                   max_steps=60000, target_acc=0.99, log_every=2000,
                   device="cpu", verbose=True):
    """Gromov-style 2-layer quadratic MLP on (a+b) mod p. Returns dict with
    embeddings XA (a-side, p x width), XB (b-side), test acc trace.
    Grokks reliably for p<=97 with defaults; if stuck at max_steps with high
    train / low test acc, increase wd or train_frac."""
    import torch
    g = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    pairs = torch.tensor([(a, b) for a in range(p) for b in range(p)])
    perm = torch.randperm(len(pairs), generator=g)
    n_tr = int(train_frac * len(pairs))
    tr, te = pairs[perm[:n_tr]], pairs[perm[n_tr:]]
    y_tr, y_te = (tr[:, 0] + tr[:, 1]) % p, (te[:, 0] + te[:, 1]) % p

    Ea = torch.nn.Parameter(torch.randn(p, width, generator=g) / np.sqrt(width))
    Eb = torch.nn.Parameter(torch.randn(p, width, generator=g) / np.sqrt(width))
    W2 = torch.nn.Parameter(torch.randn(width, p, generator=g) / np.sqrt(width))
    params = [Ea, Eb, W2]
    for prm in params: prm.data = prm.data.to(device)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    onehot = torch.eye(p, device=device)
    tr, te, y_tr, y_te = [t.to(device) for t in (tr, te, y_tr, y_te)]

    def forward(idx):
        h = (Ea[idx[:, 0]] + Eb[idx[:, 1]]) ** 2
        return h @ W2

    trace = []
    for step in range(max_steps):
        opt.zero_grad()
        logits = forward(tr)
        loss = torch.mean((logits - onehot[y_tr]) ** 2)
        loss.backward(); opt.step()
        if step % log_every == 0 or step == max_steps - 1:
            with torch.no_grad():
                acc_tr = (forward(tr).argmax(1) == y_tr).float().mean().item()
                acc_te = (forward(te).argmax(1) == y_te).float().mean().item()
            trace.append((step, loss.item(), acc_tr, acc_te))
            if verbose:
                print(f"  seed {seed} step {step:6d} loss {loss.item():.5f} "
                      f"train {acc_tr:.3f} test {acc_te:.3f}")
            if acc_te >= target_acc and acc_tr >= target_acc:
                break
    return dict(XA=Ea.detach().cpu().numpy(), XB=Eb.detach().cpu().numpy(),
                trace=trace, test_acc=trace[-1][3], seed=seed, p=p,
                _torch=dict(Ea=Ea, Eb=Eb, W2=W2, te=te, y_te=y_te,
                            forward=forward))


# ---------------- Part C: transfer of a model-space ablation ------------------
def mode_plane(X, modes, k):
    """Orthonormal basis (N x 2) of the model-space plane carrying mode k."""
    B = dict(modes)[k]
    C = B.T @ X                      # 2 x N
    Qp, _ = np.linalg.qr(C.T)        # N x 2
    return Qp

def ablate_plane(X, V):
    """Project model-space rows of X off the plane spanned by V (N x r)."""
    return X - (X @ V) @ V.T


## Train seeds past grokking
(p=59, CPU: ~1–2 min/seed; early stop when train AND test ≥ 99%. If a seed stalls
with high train / low test accuracy, raise `WD` or `TRAIN_FRAC`.)

In [ ]:
runs = {}
for s in SEEDS:
    print(f"seed {s}:")
    runs[s] = train_grokking(P, s, width=WIDTH, train_frac=TRAIN_FRAC, lr=LR,
                             wd=WD, max_steps=MAX_STEPS, log_every=1000,
                             device=DEVICE, verbose=True)
    assert runs[s]['test_acc'] >= 0.99, f"seed {s} did not grok; adjust WD/TRAIN_FRAC"
modes = fourier_modes(P)

## Stable vs unstable inventory (per seed pair)

In [ ]:
import itertools, pandas as pd
pair_rows = []
for a, b in itertools.combinations(SEEDS, 2):
    XA, XB = runs[a]['XA'], runs[b]['XA']
    tA, tB = mode_table(XA, modes), mode_table(XB, modes)
    eA = np.array([r['energy'] for r in tA if r['k'] > 0])
    eB = np.array([r['energy'] for r in tB if r['k'] > 0])
    circA = np.mean([r['svals'][1]/r['svals'][0] for r in tA
                     if r['k'] > 0 and len(r['svals']) == 2])
    rep = seed_pair_report(XA, XB, modes)
    # unstable: model-space plane overlap for the top mode, raw
    en = {r['k']: r['energy'] for r in tB if r['k'] > 0}
    ktop = max(en, key=en.get)
    Vb = mode_plane(XB, modes, ktop); Va = mode_plane(XA, modes, ktop)
    Val = np.linalg.qr(rep['Q'].T @ Va)[0]
    ov_raw = np.linalg.norm(Vb.T @ Va)**2 / 2
    ov_al  = np.linalg.norm(Vb.T @ Val)**2 / 2
    pair_rows.append(dict(pair=f"{a}-{b}", raw=rep['raw_frac'],
                          aligned=rep['aligned_frac'], floor=rep['floor_frac'],
                          excess=rep['aligned_frac'] - rep['floor_frac'],
                          energy_cv=float(eA.std()/eA.mean()),
                          circularity=float(circA),
                          plane_overlap_raw=float(ov_raw),
                          plane_overlap_aligned=float(ov_al), Q=rep['Q']))
    print(f"pair {a}-{b}: raw={rep['raw_frac']:.3f} aligned={rep['aligned_frac']:.4f} "
          f"floor={rep['floor_frac']:.4f} | plane overlap raw={ov_raw:.3f} aligned={ov_al:.3f}")
pairs = pd.DataFrame([{k: v for k, v in r.items() if k != 'Q'} for r in pair_rows])
pairs.to_csv('pred1_pairs.csv', index=False)

## P1b — the null: is the alignment test vacuous?
A singular-value-matched random matrix has the same global spectrum but random
token-space frames. If it aligned as well as a real seed, Procrustes would be doing
all the work and the result would mean nothing.

In [ ]:
rng = np.random.default_rng(0)
a = SEEDS[0]
XA = runs[a]['XA']; XB = runs[SEEDS[1]]['XA']
_, s, _ = np.linalg.svd(XB, full_matrices=False)
Ur, _ = np.linalg.qr(rng.standard_normal((P, P)))
Vr, _ = np.linalg.qr(rng.standard_normal((WIDTH, WIDTH)))
Xnull = Ur @ np.diag(s) @ Vr[:, :P].T
rep_null = seed_pair_report(XA, Xnull, modes)
rep_real = seed_pair_report(XA, XB, modes)
ratio = rep_null['aligned_frac'] / max(rep_real['aligned_frac'], 1e-12)
print(f"real pair aligned residual : {rep_real['aligned_frac']:.4f}")
print(f"matched-null residual      : {rep_null['aligned_frac']:.4f}  ({ratio:.1f}x larger)")
assert ratio > 5, "NULL TOO CLOSE: alignment test lacks discrimination here"
print("P1b PASS: the test discriminates")

## P1c — transfer pilot: interventions transfer only after gauge alignment
Ablate the model-space planes of the top-5 modes in seed B using (i) B's own planes
(positive control), (ii) A's planes raw, (iii) A's planes mapped through the gauge
alignment. Readout = mean correct-class logit margin on B's test set.

In [ ]:
import torch
b = SEEDS[1]; a = SEEDS[0]
rB = runs[b]; XA, XB = runs[a]['XA'], rB['XA']
rep = seed_pair_report(XA, XB, modes); Q = rep['Q']
tB = mode_table(XB, modes)
tt = rB['_torch']; te, y_te = tt['te'], tt['y_te']

def margin_with_XA(Xa_np):
    with torch.no_grad():
        Ea = torch.tensor(Xa_np, dtype=tt['Ea'].dtype)
        h = (Ea[te[:, 0]] + tt['Eb'][te[:, 1]]) ** 2
        logits = h @ tt['W2']
        corr = logits[torch.arange(len(y_te)), y_te].clone()
        logits[torch.arange(len(y_te)), y_te] = -1e9
        return float((corr - logits.max(1).values).mean())

en = {r['k']: r['energy'] for r in tB if r['k'] > 0}
top5 = sorted(en, key=en.get)[-5:]
Vs_self  = np.linalg.qr(np.hstack([mode_plane(XB, modes, k) for k in top5]))[0]
Vs_raw   = np.linalg.qr(np.hstack([mode_plane(XA, modes, k) for k in top5]))[0]
Vs_align = np.linalg.qr(Q.T @ Vs_raw)[0]

m_int  = margin_with_XA(XB)
m_self = margin_with_XA(ablate_plane(XB, Vs_self))
m_raw  = margin_with_XA(ablate_plane(XB, Vs_raw))
m_al   = margin_with_XA(ablate_plane(XB, Vs_align))
d_self, d_raw, d_al = m_int - m_self, m_int - m_raw, m_int - m_al
print(f"margin damage: self {d_self:+.4f} | raw {d_raw:+.4f} | aligned {d_al:+.4f}")
print(f"transfer efficiency aligned = {d_al/d_self:.3f} (require >= 0.8)")
print(f"transfer efficiency raw     = {d_raw/d_self:.3f} (require <= 0.3)")
assert d_al / d_self >= 0.8 and d_raw / d_self <= 0.3, "P1c FAILED"
print("P1c PASS")

## Interpretation (pre-committed)
- **All three pass** → seed-to-seed variability in this regime is gauge, measured
  three independent ways: (i) residual collapses to the spectral floor, (ii) matched
  null does not, (iii) interventions transfer at ~full efficiency after alignment and
  poorly before. In the paper, this closes the constructed-weights caveat with learned
  weights and makes seed variability a *consequence* of the framework.
- **P1a fails** (residual ≫ floor) → seeds differ by genuinely non-gauge structure;
  the sorting-principle framing must shrink to constructed weights only.
- **P1c fails with P1a passing** → weights are gauge-equivalent but function is keyed
  to something the alignment misses — would be the most interesting failure; investigate
  the b-side embedding and W2 before concluding anything.

**Scope reminders:** dense-symmetric regime only (see plan cell); multiplicity scaling
untestable in Z/p; sparse-regime replication (1-layer transformer) is follow-up work.